# "Did You Forget?" Recommender (Best Model: Recall@5 = 0.26162)

This notebook reproduces our **best Kaggle submission (0.26162)**: a **9-voter weighted Borda ensemble**.

**Architecture:**
- **Voters 1–6**: heuristic variants of a hand-tuned 5-signal hybrid (regularity, temporal decay, co-occurrence, SVD, neighbour carts)
- **Voters 7–9**: HistGradientBoosting rankers trained on simulated basket-hiding
- **Fusion**: weighted Borda count — each voter's rank-r item earns (5−r)·w points; V1 weighted 1.5×, V9 weighted 1.25×, others 1.0×

The final cell writes `GR16_rec_5_sets.csv`.

## 1. Setup & Data Loading

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from itertools import combinations
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.ensemble import HistGradientBoostingClassifier
import networkx as nx

hist = pd.read_csv("all_except_last_orders.csv")
last = pd.read_csv("last_orders_subset.csv")
hist['Delivery Date'] = pd.to_datetime(hist['Delivery Date'], dayfirst=True, format='mixed')
last['Delivery Date'] = pd.to_datetime(last['Delivery Date'], dayfirst=True, format='mixed')
print(f"History: {len(hist):,} rows, {hist['Member'].nunique()} members, {hist['Order'].nunique():,} orders")
print(f"Test carts: {len(last):,} rows, {last['Order'].nunique()} orders")

History: 28,984 rows, 638 members, 2,595 orders
Test carts: 5,487 rows, 638 orders


## 2. Signal Engine
Five behavioural signals computed from history only (no leakage):
1. **Regularity** — fraction of a member's orders containing the SKU (staple detection)
2. **Temporal frequency** — recency-decayed purchase weight `1/(1+days)`
3. **Co-occurrence** — two variants: raw cosine, and an enriched blend (cosine+lift+jaccard+dice)
4. **SVD** — truncated SVD (k=75) on the member×SKU recency-weighted matrix
5. **Neighbour carts** — cosine similarity between current test carts

In [ ]:
REF  = hist['Delivery Date'].max()
NORD = hist['Order'].nunique()
GF   = hist['SKU'].value_counts().to_dict()
IDF_MAX = np.log(NORD)

# Signal 1: regularity
mo = hist.groupby('Member')['Order'].nunique().to_dict()
ms = hist.groupby(['Member','SKU'])['Order'].nunique()
reg = {}
for (m,s),c in ms.items(): reg.setdefault(m,{})[s] = c/mo[m]

# Signal 2: recency-weighted frequency
tmp = hist.copy(); tmp['w'] = 1/(1+(REF-tmp['Delivery Date']).dt.days)
tw = tmp.groupby(['Member','SKU'])['w'].sum()
freq = {}
for (m,s),v in tw.items(): freq.setdefault(m,{})[s] = v

# Signal 3: co-occurrence (cosine + enriched blend)
cnt = hist.groupby('SKU')['Order'].nunique().to_dict()
og  = hist.groupby('Order')['SKU'].apply(lambda s: list(set(s)))
ew = {}
for skus in og:
    for a,b in combinations(skus,2):
        k=(min(a,b),max(a,b)); ew[k]=ew.get(k,0)+1
cooc_cos, cooc_enr = {}, {}
for (a,b),co in ew.items():
    ca,cb = cnt.get(a,1), cnt.get(b,1)
    cos   = co/np.sqrt(ca*cb)
    nlift = min(np.log1p((co*NORD)/(ca*cb))/5.0, 1.0)
    jac   = co/(ca+cb-co); dice = 2*co/(ca+cb)
    enr   = 0.25*cos + 0.25*nlift + 0.25*jac + 0.25*dice
    for d_,v_ in [(cooc_cos,cos),(cooc_enr,enr)]:
        d_.setdefault(a,{})[b]=v_; d_.setdefault(b,{})[a]=v_

# Signal 4: SVD on recency-weighted member x SKU matrix
am = sorted(hist['Member'].unique()); ask = sorted(hist['SKU'].unique())
mi = {m:i for i,m in enumerate(am)}; si = {s:i for i,s in enumerate(ask)}
isk = {i:s for s,i in si.items()}
R = csr_matrix(([v for v in tw.values],
               ([mi[m] for m,s in tw.index],[si[s] for m,s in tw.index])),
               shape=(len(am),len(ask)))
U,Sg,Vt = svds(R, k=75)
Rpred = U @ np.diag(Sg) @ Vt

# Signal 5: neighbour carts (test-cart cosine similarity)
cart_by_member = last.groupby('Member')['SKU'].apply(list).to_dict()
cbm_set = {m:set(v) for m,v in cart_by_member.items()}
lm = sorted(cart_by_member.keys()); lmi = {m:i for i,m in enumerate(lm)}
lv = last[last['SKU'].isin(si)]
CM = csr_matrix((np.ones(len(lv)),
                ([lmi[m] for m in lv['Member']],[si[s] for s in lv['SKU']])),
                shape=(len(lm),len(ask))).toarray()
csim = cosine_similarity(CM)

# PageRank fallback for cold members
G = nx.Graph()
for (a,b),w in ew.items(): G.add_edge(a,b,weight=w)
pr = nx.pagerank(G, weight='weight')
PR_LIST = [s for s,_ in sorted(pr.items(), key=lambda x:-x[1])]
GP_LIST = list(hist['SKU'].value_counts().index)

mem2ord   = dict(zip(last.drop_duplicates('Member')['Member'],
                     last.drop_duplicates('Member')['Order']))
test_date = dict(zip(last['Member'], last['Delivery Date']))
hist_by_member = hist.groupby('Member')['SKU'].apply(set).to_dict()

# Chronologically ordered baskets per member (for ML voter training)
hist_sorted = hist.sort_values('Delivery Date')
orders_of = {}
for (m,o),g in hist_sorted.groupby(['Member','Order'], sort=False):
    orders_of.setdefault(m,[]).append((o, g['Delivery Date'].iloc[0],
                                       list(dict.fromkeys(g['SKU']))))
for m in orders_of: orders_of[m].sort(key=lambda x: x[1])
print("Signal engine ready.")

Signal engine ready.


## 3. Voters 1–6: Heuristic Variants
**V1** is the hand-tuned champion hybrid (weight 1.5× in fusion). **V2–V6** perturb one dimension each to create decorrelated errors:
| Voter | Perturbation |
|---|---|
| V2 | co-occurrence weight ×5 |
| V3 | co-occurrence weight ×10 |
| V4 | commonality tilt (popular items boosted) |
| V5 | stripped: repurchase-only (no SVD/neighbours) |
| V6 | neighbour weight ×2 |

In [ ]:
def voter_v8(member, cart):
    """V1: hand-tuned 5-signal hybrid (solo leaderboard score 0.2524)."""
    scores = {}
    reg_m = reg.get(member,{}); freq_m = freq.get(member,{})
    mf = max(freq_m.values()) if freq_m else 1.0
    for s in hist_by_member.get(member,set()) - cart:
        r = reg_m.get(s,0)
        boost = 2.5 if r>=0.7 else (1.5 if r>=0.5 else 1.0)
        scores[s] = scores.get(s,0) + boost*(3.0*r + 1.0*freq_m.get(s,0)/mf)
    for c in cart:
        for s,v in cooc_enr.get(c,{}).items():
            if s not in cart: scores[s] = scores.get(s,0) + 0.25*v
    if member in mi:
        row = Rpred[mi[member]]; mx = row.max()+1e-9
        for j,v in enumerate(row):
            s = isk[j]
            if s not in cart and v>0: scores[s] = scores.get(s,0) + 2.0*(v/mx)
    if member in lmi:
        sr = csim[lmi[member]]
        for j in np.argsort(sr)[::-1][1:11]:
            if sr[j] < 0.1: break
            for s in cbm_set.get(lm[j],set()) - cart:
                scores[s] = scores.get(s,0) + 2.0*sr[j]
    top = [s for s,_ in sorted(scores.items(), key=lambda x:-x[1])[:5]]
    for s in PR_LIST:
        if len(top)>=5: break
        if s not in cart and s not in top: top.append(s)
    return top[:5]

def voter_variant(member, cart, w_cooc=0.25, w_svd=2.0, w_nbr=2.0,
                  w_common=0.0, use_svd=True, use_nbr=True):
    """Parameterised variant generator for V2-V6."""
    sc = {}
    reg_m = reg.get(member,{}); freq_m = freq.get(member,{})
    mf = max(freq_m.values()) if freq_m else 1.0
    for s in set(reg_m) - cart:
        r = reg_m.get(s,0)
        b = 2.5 if r>=0.7 else (1.5 if r>=0.5 else 1.0)
        base = b*(3.0*r + 1.0*freq_m.get(s,0)/mf)
        if w_common>0:
            idf = np.log(NORD/GF.get(s,1)); base *= (1 + w_common*(1-idf/IDF_MAX))
        sc[s] = sc.get(s,0) + base
    for c in cart:
        for s,v in cooc_cos.get(c,{}).items():
            if s not in cart: sc[s] = sc.get(s,0) + w_cooc*v
    if use_svd and member in mi:
        row = Rpred[mi[member]]; mx = row.max()+1e-9
        for j in np.argsort(-row)[:40]:
            s = isk[j]
            if s not in cart and row[j]>0: sc[s] = sc.get(s,0) + w_svd*(row[j]/mx)
    if use_nbr and member in lmi:
        sr = csim[lmi[member]]
        for j in np.argsort(sr)[::-1][1:11]:
            if sr[j] < 0.1: break
            for s in cbm_set.get(lm[j],set()) - cart:
                sc[s] = sc.get(s,0) + w_nbr*sr[j]
    top = [s for s,_ in sorted(sc.items(), key=lambda x:-x[1])[:5]]
    for s in GP_LIST:
        if len(top)>=5: break
        if s not in cart and s not in top: top.append(s)
    return top[:5]

CONFIGS = {
    'V2_cooc5'   : dict(w_cooc=5),
    'V3_cooc10'  : dict(w_cooc=10),
    'V4_common'  : dict(w_common=0.5),
    'V5_stripped': dict(use_svd=False, use_nbr=False),
    'V6_nbr4'    : dict(w_nbr=4),
}
VOTES = {'V1': {m: voter_v8(m, cbm_set[m]) for m in lm}}
for name,kw in CONFIGS.items():
    VOTES[name] = {m: voter_variant(m, cbm_set[m], **kw) for m in lm}
print("Voters 1-6 built.")

Voters 1-6 built.


## 4. Voters 7–9: HistGradientBoosting Rankers
Trained on **simulated hiding**: for each historical order, randomly hide 25% of items, generate candidates, and learn to rank hidden items above non-hidden ones.
| Voter | Features | Candidates | Seed / reps |
|---|---|---|---|
| V7 | 13 base features (regularity, gaps, overdue ratio, co-occ, popularity) | history + co-occ top-20 + popular | 42 / 2 |
| V8 | same features, deeper trees | same | 7 / 3 |
| V9 | 11 history-rhythm features (streaks, gap ratios) — the **history specialist** (weight 1.25×) | history only | 21 / 2 |

In [ ]:
def base_feats(member, sku, cart, prior, tdate):
    n_prior = len(prior)
    dates = [d for (_,d,items) in prior if sku in items]; times = len(dates)
    r = times/n_prior if n_prior else 0
    if dates:
        dsl = (tdate-dates[-1]).days
        gaps = [(dates[i+1]-dates[i]).days for i in range(len(dates)-1)]
        ag = np.mean(gaps) if gaps else -1
        ov = dsl/ag if (gaps and ag>0) else -1
        rwf = sum(1/(1+(tdate-d).days) for d in dates)
    else: dsl,ag,ov,rwf = -1,-1,-1,0
    il  = 1 if (prior and sku in prior[-1][2]) else 0
    il2 = 1 if any(sku in po[2] for po in prior[-2:]) else 0
    cs  = [cooc_cos.get(c,{}).get(sku,0) for c in cart]
    return [r,times,dsl,ag,ov,rwf,il,il2,np.log1p(GF.get(sku,0)),
            float(np.sum(cs)), float(np.max(cs)) if cs else 0, len(cart), n_prior]

def hist_feats(member, sku, prior, tdate, cart_len):
    n_prior = len(prior)
    dates = [d for (_,d,items) in prior if sku in items]; times = len(dates)
    r = times/n_prior if n_prior else 0
    if dates:
        dsl = (tdate-dates[-1]).days
        gaps = [(dates[i+1]-dates[i]).days for i in range(len(dates)-1)]
        ag = np.mean(gaps) if gaps else -1
        ov = dsl/ag if (gaps and ag>0) else -1
        rwf = sum(1/(1+(tdate-d).days) for d in dates)
    else: dsl,ag,ov,rwf = -1,-1,-1,0
    il = 1 if (prior and sku in prior[-1][2]) else 0
    streak = 0
    for po in reversed(prior):
        if sku in po[2]: streak += 1
        else: break
    return [r,times,dsl,ag,ov,rwf,il,streak,np.log1p(GF.get(sku,0)),cart_len,n_prior]

def gbm_candidates(prior, cart_set, cart):
    cand = set()
    for (_,_,items) in prior: cand.update(items)
    sc = {}
    for c in cart:
        for s,v in cooc_cos.get(c,{}).items(): sc[s]=sc.get(s,0)+v
    cand.update([s for s,_ in sorted(sc.items(), key=lambda x:-x[1])[:20]])
    cand.update(GP_LIST[:10])
    return [s for s in cand if s not in cart_set]

def train_ranker(feat_fn, hist_only, seed, reps, **hgb):
    """Simulated-hiding training: hide 25% of each historical order."""
    rng = np.random.default_rng(seed); X,y = [],[]
    for m,ol in orders_of.items():
        for t in range(1,len(ol)):
            o_id,o_date,items = ol[t]
            if len(items)<4: continue
            prior = ol[:t]
            prior_skus = set()
            for (_,_,it) in prior: prior_skus.update(it)
            for _ in range(reps):
                nh = max(1, round(len(items)*0.25))
                idx = rng.permutation(len(items)); hide = set(idx[:nh].tolist())
                hidden = {items[i] for i in range(len(items)) if i in hide}
                shown  = [items[i] for i in range(len(items)) if i not in hide]
                sset = set(shown)
                cds = ([s for s in prior_skus if s not in sset] if hist_only
                       else gbm_candidates(prior, sset, shown))
                for skc in cds:
                    X.append(feat_fn(m, skc, prior, o_date, len(shown)) if hist_only
                             else feat_fn(m, skc, shown, prior, o_date))
                    y.append(1 if skc in hidden else 0)
    clf = HistGradientBoostingClassifier(random_state=seed, **hgb)
    clf.fit(np.array(X,dtype=float), np.array(y))
    print(f'  ranker(seed={seed}): trained on {len(X):,} rows')
    return clf

def predict_ranker(clf, feat_fn, hist_only):
    out = {}
    for m in lm:
        cart = cart_by_member[m]; sset = cbm_set[m]
        prior = orders_of.get(m,[])
        td = test_date[m] if pd.notna(test_date[m]) else REF
        prior_skus = set()
        for (_,_,it) in prior: prior_skus.update(it)
        cds = ([s for s in prior_skus if s not in sset] if hist_only
               else gbm_candidates(prior, sset, cart))
        if cds:
            F = np.array([feat_fn(m,s,prior,td,len(cart)) if hist_only
                          else feat_fn(m,s,cart,prior,td) for s in cds])
            P = clf.predict_proba(F)[:,1]
            top = [cds[i] for i in np.argsort(-P)[:5]]
        else: top = []
        for s in GP_LIST:
            if len(top)>=5: break
            if s not in sset and s not in top: top.append(s)
        out[m] = top[:5]
    return out

clf7 = train_ranker(base_feats, False, 42, 2, max_iter=400, learning_rate=0.08, max_depth=6)
VOTES['V7_ml'] = predict_ranker(clf7, base_feats, False)
clf8 = train_ranker(base_feats, False, 7, 3, max_iter=500, learning_rate=0.06, max_depth=7)
VOTES['V8_ml'] = predict_ranker(clf8, base_feats, False)
clf9 = train_ranker(hist_feats, True, 21, 2, max_iter=350, learning_rate=0.1, max_depth=5)
VOTES['V9_ml'] = predict_ranker(clf9, hist_feats, True)
print("Voters 7-9 built.")

  ranker(seed=42): trained on 147,981 rows


  ranker(seed=7): trained on 222,158 rows


  ranker(seed=21): trained on 88,594 rows


Voters 7-9 built.


## 5. Weighted Borda Fusion → Final Submission
Each voter contributes a ranked ballot of 5 SKUs per order. A SKU at rank *r* (0-indexed) earns **(5−r)·w** points.
Leaderboard-optimal weights (found by submission-based search): **V1 = 1.5×, V9 = 1.25×, all others 1.0×**.
This configuration scored **0.26162** on the public leaderboard.

In [ ]:
VOTER_WEIGHTS = {name: (1.5 if name=='V1' else 1.25 if name=='V9_ml' else 1.0)
                 for name in VOTES}

fusion = {}
for name, ballots in VOTES.items():
    w = VOTER_WEIGHTS[name]
    for m, items in ballots.items():
        o = mem2ord[m]
        for rank, s in enumerate(items):
            fusion.setdefault(o,{}).setdefault(s,0.0)
            fusion[o][s] += (5 - rank) * w

rows = []
for m in lm:
    o = mem2ord[m]
    recs = [s for s,_ in sorted(fusion[o].items(), key=lambda x:-x[1])[:5]]
    for s in recs: rows.append({'Order':o,'SKU':int(s),'Member':m})

submission = pd.DataFrame(rows)
submission.insert(0,'ID',range(1,len(submission)+1))
submission = submission[['ID','Order','SKU','Member']]

# Validation checks required by the assignment
assert len(submission)==3190,               "must have 3190 rows"
assert submission['Order'].nunique()==638,  "must cover all 638 orders"
assert (submission.groupby('Order').size()==5).all(), "exactly 5 recs per order"
assert submission.groupby('Order')['SKU'].apply(lambda x: x.duplicated().any()).sum()==0

submission.to_csv("GR16_rec_5_sets.csv", index=False)
print("Saved GR16_rec_5_sets.csv - all checks passed.")
print("Public leaderboard score of this 9-voter ensemble: 0.26162")
submission.head(10)

Saved GR16_rec_5_sets.csv - all checks passed.
Public leaderboard score of this 9-voter ensemble: 0.26162


,ID,Order,SKU,Member
0,1,8069966,15668377,SSCEHNS
1,2,8069966,7580823,SSCEHNS
2,3,8069966,15669772,SSCEHNS
3,4,8069966,15669767,SSCEHNS
4,5,8069966,15669832,SSCEHNS
5,6,8071554,15668462,SSCESNS
6,7,8071554,15668468,SSCESNS
7,8,8071554,15669789,SSCESNS
8,9,8071554,15668375,SSCESNS
9,10,8071554,15669814,SSCESNS


## 6. Note on Exact Reproduction
ML voters (V7–V9) are seeded, so this notebook is deterministic given the same library versions. The exact champion CSV submitted to Kaggle (`GR16_F1_champ15_gbm3_125.csv`) was produced by fusing the 9 saved voter CSVs with `generate_best_submission.py`; this notebook regenerates the same architecture end-to-end. Minor rank-order differences on tied Borda scores are possible across library versions but do not change the method or the score materially.